[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ashkanhszd/m8_data_science_module/blob/main/soluciones/M8_8.22_24_soluciones.ipynb)

# SOLUCIONES · M8_8.22–M8_8.24 · NumPy, pandas y Matplotlib

**Máster en Hidrogeología y Modelación (UB) · Módulo 8: Herramientas y Data Science**

### Objetivos de la tarde

1. Calcular con *arrays* de NumPy sin escribir bucles (vectorización), seleccionar valores con máscaras y tratar valores ausentes.
2. Leer, inspeccionar, filtrar, unir y resumir tablas con pandas.
3. Construir figuras científicas legibles con Matplotlib: series temporales, hidrogramas, dispersión, histogramas y un mapa de pozos.
4. Integrar las tres bibliotecas para interpretar un **ensayo de bombeo** con el método de Cooper–Jacob.
5. Resolver dudas consultando la documentación oficial.

### Plan de la tarde

| Sesión | Contenido | Tiempo aprox. |
|---|---|---|
| **M8_8.22** | NumPy: arrays, vectorización, máscaras, `NaN`, matrices y ejes | 50 min |
| *Descanso* | | 10 min |
| **M8_8.23** | pandas: leer, inspeccionar, seleccionar, unir, agrupar, reorganizar y guardar tablas | 60 min |
| *Descanso* | | 10 min |
| **M8_8.24** | Matplotlib: anatomía de una figura, series, hidrograma, dispersión, histograma, mapa · caso: ensayo de bombeo | 60 min |

### Los datos del curso

Durante todo el módulo usamos el mismo conjunto de datos sintéticos: **12 pozos** en **3 acuíferos** (aluvial, carbonatado y detrítico), con medidas mensuales de 2021 a 2025, **2 estaciones meteorológicas**, análisis químicos y un ensayo de bombeo. La descripción está en `data/input/README.md` y el significado de cada columna, en `data/input/diccionario_datos.csv`.

> Como ocurre con los datos reales, **estos datos contienen errores**. Hoy nos encontraremos algunos. En M8_8.27 los trataremos de forma sistemática.

Iconos: 💡 idea clave · 📖 documentación · ✏️ ejercicio · ✅ comprobación · ⚠️ error frecuente

## 0. Preparación

Guarda una copia en Drive (*Archivo → Guardar una copia en Drive*) y ejecuta las dos celdas siguientes.

In [ ]:
# ==================================================================
# PREPARACIÓN DEL ENTORNO · ejecuta siempre esta celda en primer lugar
# ==================================================================
# · En Google Colab: descarga (clona) el repositorio del curso y se sitúa en él.
# · En un ordenador propio: busca la carpeta del repositorio.
# No hace falta entender cada línea todavía; la estructura se explica en M8_8.20.

import os
import subprocess
import sys
from pathlib import Path

URL_REPOSITORIO = "https://github.com/Ashkanhszd/m8_data_science_module.git"
EN_COLAB = "google.colab" in sys.modules

if EN_COLAB:
    ROOT = Path("/content/m8_data_science_module")
    if not ROOT.exists():
        print("Descargando el repositorio del curso...")
        subprocess.run(["git", "clone", "--depth", "1", URL_REPOSITORIO, str(ROOT)], check=True)
else:
    ROOT = None
    for carpeta in [Path.cwd(), *Path.cwd().parents]:
        if (carpeta / "data" / "input").exists():
            ROOT = carpeta
            break
    if ROOT is None:
        raise FileNotFoundError("No se encuentra data/input: abre el notebook desde la carpeta del repositorio.")

os.chdir(ROOT)
INPUT = ROOT / "data" / "input"          # datos originales (no se modifican)
DATABASE = ROOT / "data" / "database"    # base de datos SQLite
OUTPUT = ROOT / "data" / "output"        # resultados que generamos
for subcarpeta in ["tables", "figures", "logs"]:
    (OUTPUT / subcarpeta).mkdir(parents=True, exist_ok=True)

print("Entorno:", "Google Colab" if EN_COLAB else "ordenador local")
print("Carpeta del repositorio:", ROOT)
print("Python", sys.version.split()[0])

In [ ]:
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

print("NumPy", np.__version__, "| pandas", pd.__version__, "| Matplotlib", matplotlib.__version__)

# Un color fijo para cada acuífero, que usaremos en todas las figuras del módulo
COLORES_ACUIFERO = {"Aluvial": "#2a78d6", "Carbonatado": "#eb6834", "Detritico": "#1baf7a"}

---
# M8_8.22 · Computación científica con NumPy

## 1. ¿Por qué NumPy?

En M8_8.21 calculamos la cota piezométrica de varios pozos con un bucle `for`. Funciona, pero con miles de valores es lento y el código se hace largo. **NumPy** introduce el **array**: una colección de números del mismo tipo con la que se puede operar **de una sola vez**, como si fuera un único número. Esto se llama **vectorización**.

In [ ]:
profundidades_m = np.array([9.28, 8.80, 8.45, 8.78, 9.73, 10.42])   # P03, enero a junio de 2023
cota_terreno_m = 105.9

cotas_piezometricas_m = cota_terreno_m - profundidades_m    # sin bucle: se resta a cada elemento
print(cotas_piezometricas_m)

> ⚠️ **Error frecuente:** una lista y un array **no se comportan igual** con los operadores.

In [ ]:
lista = [8.8, 9.3]
array = np.array([8.8, 9.3])

print("lista * 2 →", lista * 2)     # ¡repite la lista!
print("array * 2 →", array * 2)     # multiplica cada valor

## 2. Forma, tamaño y tipo

| Atributo | Qué indica |
|---|---|
| `ndim` | número de dimensiones (1 = vector, 2 = matriz…) |
| `shape` | tamaño en cada dimensión |
| `size` | número total de elementos |
| `dtype` | tipo de los datos (`float64`, `int64`, `bool`…) |

Estos son **atributos** del objeto array: se consultan sin paréntesis.

In [ ]:
print("ndim :", profundidades_m.ndim)
print("shape:", profundidades_m.shape)
print("size :", profundidades_m.size)
print("dtype:", profundidades_m.dtype)

## 3. Índices y cortes

Funcionan igual que en las listas: el primer elemento es el `0`, el último el `-1`, y en un corte `inicio:fin` el elemento `fin` **no** se incluye.

In [ ]:
print("Enero:", profundidades_m[0])
print("Junio:", profundidades_m[-1])
print("Primer trimestre:", profundidades_m[0:3])

## 4. Funciones matemáticas y estadísticas

NumPy incluye las funciones matemáticas habituales. Todas trabajan elemento a elemento:

| Función | Uso |
|---|---|
| `np.mean`, `np.median`, `np.min`, `np.max`, `np.std` | resúmenes |
| `np.sqrt`, `np.log10`, `np.log`, `np.exp` | funciones matemáticas |
| `np.round(x, 2)` | redondeo |
| `np.pi` | la constante $\pi$ |

Ejemplo: conversión de caudales de L/s a m³/día (×86.4) y área de la sección de pozos de distinto diámetro, $A = \pi r^2$.

In [ ]:
caudales_l_s = np.array([5, 12, 20, 35])
print("Caudales (m³/día):", caudales_l_s * 86.4)

diametros_m = np.array([0.15, 0.20, 0.30])
areas_m2 = np.pi * (diametros_m / 2) ** 2
print("Áreas (m²):", np.round(areas_m2, 4))

print("Media de las profundidades:", np.mean(profundidades_m).round(2), "m")

## 5. Comparaciones y máscaras booleanas

Una comparación aplicada a un array devuelve un array de `True`/`False`, que llamamos **máscara**. La máscara sirve para **seleccionar** los valores que cumplen la condición. Como `True` vale 1 y `False` vale 0, sumar la máscara **cuenta** cuántos valores la cumplen.

In [ ]:
umbral_m = 9.0
mascara = profundidades_m > umbral_m

print("Máscara:", mascara)
print("Valores que superan el umbral:", profundidades_m[mascara])
print("¿Cuántos?", mascara.sum())

# np.where: elegir un valor u otro según una condición
print(np.where(profundidades_m > umbral_m, "profundo", "normal"))

## 6. Valores ausentes: `NaN`

Si falta una medida, se representa con `np.nan`. Cualquier operación con un `NaN` da `NaN`, lo que nos avisa de que hay datos ausentes. Las funciones `np.nan…` los ignoran.

> 💡 Ignorar un ausente permite calcular, pero **no resuelve por qué falta**. Esa decisión la estudiaremos en M8_8.27.

In [ ]:
con_ausente = np.array([9.28, 8.80, np.nan, 8.78])

print("np.mean    →", np.mean(con_ausente))
print("np.nanmean →", np.nanmean(con_ausente))
print("¿Cuántos ausentes?", np.isnan(con_ausente).sum())

## 7. Matrices y ejes

Un array de 2 dimensiones es una **matriz**. Aquí, cada **fila** es un pozo y cada **columna** es un mes de 2023:

```text
                 ene    feb    mar   ...   dic        axis=1 →  (recorre las columnas:
      P01   [   7.96   7.38   7.12  ...   7.72 ]                 un resultado por fila / pozo)
      P05   [  25.78  25.88  25.55  ...  26.22 ]
      P08   [  15.64  15.46  15.18  ...  15.78 ]
                ↓ axis=0  (recorre las filas: un resultado por columna / mes)
```

In [ ]:
matriz = np.array([
    [7.96, 7.38, 7.12, 7.50, 7.59, 7.83, 7.92, 7.85, 8.07, 8.02, 8.15, 7.72],       # P01
    [25.78, 25.88, 25.55, 25.67, 25.76, 25.18, 25.57, 25.75, 25.94, 26.03, 26.13, 26.22],  # P05
    [15.64, 15.46, 15.18, 14.82, 15.01, 15.18, 15.34, 15.40, 15.53, 15.68, 15.62, 15.78],  # P08
])

print("shape:", matriz.shape, "→ 3 pozos × 12 meses")
print("Media de cada pozo (axis=1):", matriz.mean(axis=1).round(2))
print("Media de cada mes  (axis=0):", matriz.mean(axis=0).round(2))
print("Valor de P05 en marzo:", matriz[1, 2])

> 💡 **Conexión con M8_8.34–36.** Un modelo numérico de flujo guarda los niveles calculados en arrays de **3 dimensiones**: `(capa, fila, columna)`, y a menudo de 4 si añadimos el tiempo. Todo lo que hemos visto (índices, cortes, máscaras, ejes) se aplica igual.

> 📖 **Documentación:** la guía para principiantes de NumPy es muy recomendable: https://numpy.org/doc/stable/user/absolute_beginners.html. En la referencia de la API se encuentra cualquier función, por ejemplo `np.argmax`: https://numpy.org/doc/stable/reference/generated/numpy.argmax.html

### ✏️ Ejercicio 1 · El año 2023 en el pozo P03

`p03_2023` contiene la profundidad mensual del nivel en P03 (cota del terreno 105.9 m) durante 2023.

1. Calcula la cota piezométrica de cada mes (`h_p03`).
2. Cuenta cuántos meses la profundidad superó **10 m** (`meses_mas_10`).
3. Busca en la documentación una función que devuelva la **posición** del valor máximo y úsala para saber en qué mes (1 = enero) el nivel estuvo más profundo (`mes_mas_profundo`).
4. Calcula la oscilación anual (máximo − mínimo) en metros (`oscilacion_m`).

In [ ]:
p03_2023 = np.array([9.28, 8.80, 8.45, 8.78, 9.73, 10.42, 10.74, 10.64, 10.24, 9.71, 9.65, 9.30])

In [ ]:
h_p03 = 105.9 - p03_2023
meses_mas_10 = (p03_2023 > 10).sum()
mes_mas_profundo = np.argmax(p03_2023) + 1      # +1 porque los índices empiezan en 0
oscilacion_m = p03_2023.max() - p03_2023.min()

print(h_p03, meses_mas_10, mes_mas_profundo, oscilacion_m)

In [ ]:
# ✅ Comprobación
assert np.allclose(h_p03, 105.9 - p03_2023)
assert meses_mas_10 == 4, "Usa una máscara (p03_2023 > 10) y súmala"
assert mes_mas_profundo == 7, "np.argmax devuelve la posición (desde 0): ¿qué mes es?"
assert abs(oscilacion_m - 2.29) < 1e-6
print("✔ Correcto. El nivel más profundo se da en julio, con el bombeo para riego en pleno verano.")

---
# M8_8.23 · Manipulación y análisis de datos con pandas

## 8. Series y DataFrame

**pandas** trabaja con tablas:

- un **DataFrame** es una tabla con filas, columnas con nombre y un **índice** que identifica cada fila;
- cada columna es una **Series**: un array de NumPy con etiquetas.

La lista de diccionarios del final de M8_8.21 se convierte directamente en un DataFrame:

In [ ]:
registros = [
    {"id_pozo": "P01", "acuifero": "Aluvial", "profundidad_m": 7.85},
    {"id_pozo": "P04", "acuifero": "Carbonatado", "profundidad_m": 21.40},
    {"id_pozo": "P08", "acuifero": "Detritico", "profundidad_m": 15.10},
]
tabla = pd.DataFrame(registros)
tabla

## 9. Leer archivos CSV

En la práctica las tablas se leen desde archivos. `pd.read_csv` lee un archivo de texto separado por comas. Con `parse_dates` indicamos qué columnas contienen fechas, para que pandas las trate como fechas y no como texto.

In [ ]:
pozos = pd.read_csv(INPUT / "pozos.csv")
mediciones = pd.read_csv(INPUT / "mediciones.csv", parse_dates=["fecha"])

pozos

## 10. Inspección inicial: lo primero que se hace siempre

| Instrucción | Qué muestra |
|---|---|
| `df.head()` / `df.tail()` | primeras / últimas filas |
| `df.shape` | (número de filas, número de columnas) |
| `df.columns` | nombres de las columnas |
| `df.dtypes` | tipo de cada columna |
| `df.info()` | resumen: tipos y valores no nulos |
| `df.describe()` | estadísticos de las columnas numéricas |

In [ ]:
print("Dimensiones:", mediciones.shape)
mediciones.head()

In [ ]:
mediciones.info()

In [ ]:
mediciones.describe()

> 💡 **Mira siempre `describe()` con ojos de hidrogeólogo.** ¿Una profundidad máxima de 270 m en pozos que no llegan a 100 m? ¿Una conductividad mínima de 0.97 µS/cm, menor que la del agua destilada? `describe()` es la primera herramienta de control de calidad. Apunta estas dos rarezas: las resolveremos en M8_8.27.

## 11. Seleccionar columnas y filas

- `df["columna"]` devuelve una columna (Series).
- `df[["col1", "col2"]]` devuelve varias columnas (DataFrame). Fíjate en los **dobles corchetes**.
- `df.loc[filas, columnas]` selecciona por **etiquetas** o por **condiciones**.
- `df.iloc[filas, columnas]` selecciona por **posición** (números).

In [ ]:
mediciones[["id_pozo", "fecha", "profundidad_nivel_m"]].head(3)

In [ ]:
mediciones.iloc[0:3]

## 12. Filtrar filas con condiciones

Una buena práctica para principiantes es escribir cada condición en una variable con nombre. Las condiciones se combinan con `&` (y) y `|` (o), nunca con `and` / `or`.

**Pregunta:** ¿qué profundidades se midieron en P03 en el verano de 2023?

In [ ]:
es_p03 = mediciones["id_pozo"] == "P03"
es_2023 = mediciones["fecha"].dt.year == 2023
es_verano = mediciones["fecha"].dt.month.isin([6, 7, 8])

mediciones.loc[es_p03 & es_2023 & es_verano, ["id_pozo", "fecha", "profundidad_nivel_m"]]

`.dt` da acceso a las partes de una fecha (`.dt.year`, `.dt.month`, `.dt.day`…). Solo funciona si la columna es de tipo fecha, y por eso usamos `parse_dates` al leer.

## 13. Ordenar

In [ ]:
mediciones.sort_values("conductividad_uScm", ascending=False).head()

## 14. Unir tablas con `merge`

`mediciones` no dice a qué acuífero pertenece cada pozo ni cuál es su cota: esa información está en `pozos`. Las dos tablas comparten la columna **`id_pozo`**, que es la **clave** que las relaciona.

```text
 mediciones (muchas filas por pozo)        pozos (una fila por pozo)
 id_pozo  fecha       profundidad          id_pozo  acuifero  cota_terreno_m
 P01      2021-01-01  7.91        ──┐      P01      Aluvial   112.4
 P01      2021-02-01  7.84        ──┼────► ...
 ...                                │
```

- `how="left"` conserva **todas** las mediciones, aunque no encuentren su pozo.
- `validate="many_to_one"` comprueba que cada pozo aparece una sola vez en `pozos` (muchas mediciones → un pozo).

In [ ]:
datos = mediciones.merge(pozos, on="id_pozo", how="left", validate="many_to_one")

# Nueva columna calculada de forma vectorizada: h = z - d
datos["cota_piezometrica_m"] = datos["cota_terreno_m"] - datos["profundidad_nivel_m"]

datos[["id_pozo", "fecha", "acuifero", "cota_terreno_m", "profundidad_nivel_m", "cota_piezometrica_m"]].head()

Una comprobación que conviene hacer siempre después de unir tablas: **¿alguna medición se ha quedado sin pozo?**

In [ ]:
sin_pozo = datos["acuifero"].isna()
print("Mediciones sin pozo asociado:", sin_pozo.sum())
datos.loc[sin_pozo, ["id_pozo", "fecha", "profundidad_nivel_m"]]

Hay una medición del pozo **`P1`**, que no existe en la tabla de pozos. Probablemente sea `P01` mal escrito. Anótalo también para M8_8.27.

## 15. Agrupar y resumir con `groupby`

`groupby` divide la tabla en grupos, aplica un cálculo a cada grupo y reúne los resultados. Se lee así: *agrupa por acuífero, toma la profundidad y calcula…*

In [ ]:
datos.groupby("acuifero")["profundidad_nivel_m"].agg(["count", "mean", "min", "max"]).round(2)

In [ ]:
# Media anual por acuífero: agrupar por dos columnas
datos["año"] = datos["fecha"].dt.year
media_anual = datos.groupby(["acuifero", "año"])["profundidad_nivel_m"].mean().round(2)
media_anual

## 16. Unir por dos claves: la precipitación

La precipitación no es una propiedad del pozo, sino de la **estación meteorológica**. Por eso está en otra tabla. Cada pozo tiene una estación de referencia (`id_estacion`). Para asignar a cada medición la lluvia de su mes hay que unir por **dos columnas**: estación **y** fecha.

In [ ]:
precipitacion = pd.read_csv(INPUT / "precipitacion.csv", parse_dates=["fecha"])

datos = datos.merge(precipitacion, on=["id_estacion", "fecha"], how="left", validate="many_to_one")
datos[["id_pozo", "fecha", "id_estacion", "precipitacion_mm", "profundidad_nivel_m"]].head()

## 17. Formato largo y formato ancho

- **Formato largo**: una fila por observación (pozo + fecha). Es el formato de `mediciones` y el más cómodo para filtrar y agrupar.
- **Formato ancho**: una columna por pozo. Es cómodo para comparar series.

`pivot` pasa de largo a ancho. Ejecuta la celda y **lee el error**:

In [ ]:
try:
    ancho = mediciones.pivot(index="fecha", columns="id_pozo", values="profundidad_nivel_m")
except ValueError as error:
    print("ValueError:", error)

`pivot` exige que cada combinación fecha–pozo aparezca **una sola vez**. El error revela que hay **duplicados** en los datos. Veamos cuáles:

In [ ]:
repetidos = mediciones.duplicated(subset=["id_pozo", "fecha"], keep=False)
mediciones.loc[repetidos].sort_values(["id_pozo", "fecha"])

Aparecen dos **duplicados exactos** (la misma fila dos veces) y un caso más preocupante en **P05**: la misma fecha con **dos valores distintos**. ¿Cuál es el correcto? Lo decidiremos en M8_8.27.

`pivot_table` sí funciona porque **agrega** los repetidos (por defecto, calcula la media).

> ⚠️ `pivot_table` no da error, pero **oculta** el problema: el valor de P05 en mayo de 2024 sería la media de dos medidas incompatibles. No uses una función para hacer desaparecer un error sin entenderlo antes.

In [ ]:
ancho = mediciones.pivot_table(index="fecha", columns="id_pozo", values="profundidad_nivel_m", aggfunc="mean")
ancho.head()

## 18. Leer Excel y guardar resultados

`pd.read_excel` lee una hoja de un libro Excel (`sheet_name`). Con `pd.ExcelFile` se puede ver qué hojas contiene. Los resultados se guardan en `data/output`, **nunca** en `data/input`.

In [ ]:
ruta_excel = INPUT / "datos_hidrogeologicos.xlsx"
print("Hojas:", pd.ExcelFile(ruta_excel).sheet_names)

pozos_excel = pd.read_excel(ruta_excel, sheet_name="pozos")
print("¿Mismos valores que pozos.csv?", (pozos_excel == pozos).all().all())
print("¿Mismos tipos de dato?", (pozos_excel.dtypes == pozos.dtypes).all())
print(pd.DataFrame({"CSV": pozos.dtypes, "Excel": pozos_excel.dtypes}))

media_anual.to_csv(OUTPUT / "tables" / "profundidad_media_anual_por_acuifero.csv")
print("Guardado en", OUTPUT / "tables")

Los valores son idénticos, pero **no los tipos**: en Excel, `30.0` se guarda como el número `30` y pandas lo lee como entero (`int64`). Es un detalle pequeño, pero explica por qué dos tablas "iguales" pueden no serlo para el ordenador, y por qué conviene revisar siempre `dtypes` al cambiar de formato.

### ✏️ Ejercicio 2 · Leer un archivo con otro formato (consultando la documentación)

El equipo de campo ha enviado las medidas del primer trimestre de 2026 en `campana_2026_T1.csv`. El archivo se creó con una configuración regional española y **no tiene el mismo formato** que los demás. Mira primero cómo es:

In [ ]:
print((INPUT / "campana_2026_T1.csv").read_text()[:300])

Si lo lees con `pd.read_csv` sin más, el resultado es incorrecto (pruébalo). Consulta la documentación de `read_csv` (https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html o `pd.read_csv?`) para encontrar los parámetros que indican:

1. el **separador** de columnas;
2. el carácter **decimal**;
3. los valores que deben interpretarse como **ausentes** (el equipo de campo usa `-9999`).

Después, convierte la columna `Fecha` a fecha con `pd.to_datetime`. **Cuidado:** `01/02/2026` es el 1 de febrero, no el 2 de enero. Busca en la documentación de `pd.to_datetime` cómo indicar el formato (https://pandas.pydata.org/docs/reference/api/pandas.to_datetime.html).

In [ ]:
campana = pd.read_csv(INPUT / "campana_2026_T1.csv", sep=";", decimal=",", na_values=[-9999])
campana["Fecha"] = pd.to_datetime(campana["Fecha"], format="%d/%m/%Y")   # también: dayfirst=True

campana.head()

In [ ]:
# ✅ Comprobación
assert campana.shape == (36, 4), "¿Has indicado el separador correcto?"
assert pd.api.types.is_float_dtype(campana["Prof. nivel (m)"]), "¿Has indicado el separador decimal?"
assert campana["Prof. nivel (m)"].isna().sum() == 1, "¿Has indicado que -9999 es un valor ausente?"
assert set(campana["Fecha"].dt.month) == {1, 2, 3}, "Revisa el formato de fecha: día/mes/año"
print("✔ Correcto. En M8_8.33 incorporaremos esta campaña al flujo de análisis.")

### ✏️ Ejercicio 3 · ¿Qué acuífero ha sufrido más la sequía?

Entre 2022 y 2024 hubo una sequía. Con la tabla `datos`:

1. calcula la profundidad media por acuífero en **2021** y en **2024** (`media_2021`, `media_2024`, dos Series indexadas por acuífero);
2. calcula el descenso `descenso_m = media_2024 - media_2021` (en metros);
3. ¿qué acuífero descendió más? Guarda su nombre en `acuifero_mas_afectado` (pista: `idxmax`).

*Aviso:* la tabla contiene todavía los errores que hemos visto. Para esta comparación, filtra primero las profundidades mayores que la profundidad del pozo (`profundidad_pozo_m`).

In [ ]:
datos_validos = datos.loc[datos["profundidad_nivel_m"] < datos["profundidad_pozo_m"]]

media_2021 = datos_validos.loc[datos_validos["año"] == 2021].groupby("acuifero")["profundidad_nivel_m"].mean()
media_2024 = datos_validos.loc[datos_validos["año"] == 2024].groupby("acuifero")["profundidad_nivel_m"].mean()
descenso_m = media_2024 - media_2021
acuifero_mas_afectado = descenso_m.idxmax()

print(descenso_m.round(2))
print("Más afectado:", acuifero_mas_afectado)

In [ ]:
# ✅ Comprobación
assert acuifero_mas_afectado == "Carbonatado"
assert 2.5 < descenso_m["Carbonatado"] < 3.5
print("✔ Correcto. El acuífero carbonatado, con mucha capacidad de almacenamiento y una respuesta lenta,")
print("  acumula el déficit de varios años de sequía.")

---
# M8_8.24 · Visualización científica con Matplotlib

## 19. Anatomía de una figura

- La **figura** (`fig`) es el lienzo completo.
- Los **ejes** (`ax`) son cada zona de dibujo, con su eje x, su eje y, título y leyenda.

La forma recomendada de trabajar es crear ambos con `plt.subplots()` y dar órdenes al objeto `ax`:

```python
fig, ax = plt.subplots(figsize=(ancho, alto))   # tamaño en pulgadas
ax.plot(x, y)                                   # dibujar
ax.set_xlabel("...")  ;  ax.set_ylabel("... (unidad)")  ;  ax.set_title("...")
```

> 📖 **Documentación:** la galería de Matplotlib (https://matplotlib.org/stable/gallery/) contiene cientos de ejemplos con su código. La forma más eficaz de aprender es **buscar un ejemplo parecido a lo que quieres y adaptarlo**. La figura "Anatomy of a figure" muestra el nombre de cada elemento: https://matplotlib.org/stable/gallery/showcase/anatomy.html

## 20. Serie temporal de un pozo

La profundidad se mide hacia abajo desde la superficie. Si **invertimos el eje y**, la figura se lee de forma intuitiva: cuando la línea baja, el nivel del agua baja.

In [ ]:
p05 = datos.loc[datos["id_pozo"] == "P05"].sort_values("fecha")

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(p05["fecha"], p05["profundidad_nivel_m"], color=COLORES_ACUIFERO["Carbonatado"],
        linewidth=1.8, marker="o", markersize=3)
ax.invert_yaxis()
ax.set_title("P05 (acuífero carbonatado): profundidad del nivel")
ax.set_xlabel("Fecha")
ax.set_ylabel("Profundidad del nivel (m)")
ax.grid(alpha=0.3)
plt.show()

## 21. Varios pozos: un panel por acuífero

Si dibujamos los 12 pozos en un solo gráfico, las líneas se mezclan y la figura se vuelve ilegible. Es mejor usar **paneles pequeños** (*small multiples*), uno por acuífero, que comparten el eje x (`sharex=True`). Cada pozo se identifica con una etiqueta al final de su línea.

In [ ]:
# Excluimos el valor imposible (más profundo que el propio pozo), pero conservamos los ausentes (NaN)
# para que los huecos de datos se vean como huecos en la línea
datos_graf = datos.loc[~(datos["profundidad_nivel_m"] > datos["profundidad_pozo_m"])].sort_values("fecha")

fig, ejes = plt.subplots(3, 1, figsize=(11, 9), sharex=True)

for ax, (acuifero, grupo) in zip(ejes, datos_graf.groupby("acuifero")):
    for id_pozo, serie in grupo.groupby("id_pozo"):
        serie = serie.sort_values("fecha")
        ax.plot(serie["fecha"], serie["profundidad_nivel_m"], color=COLORES_ACUIFERO[acuifero], linewidth=1.5)
        ax.text(serie["fecha"].iloc[-1], serie["profundidad_nivel_m"].iloc[-1], "  " + id_pozo,
                va="center", fontsize=9)
    ax.invert_yaxis()
    ax.set_title(acuifero, loc="left")
    ax.set_ylabel("Profundidad (m)")
    ax.grid(alpha=0.3)

ejes[-1].set_xlabel("Fecha")
fig.suptitle("Profundidad del nivel por acuífero, 2021–2025")
fig.tight_layout()
plt.show()

**¿Qué se observa?** La sequía de 2022–2024 y la recuperación de 2025 se ven en los tres acuíferos, pero con respuestas diferentes:

- el **aluvial** oscila cada año y los pozos de riego (P03, P10) bajan en verano;
- el **carbonatado** responde lentamente y acumula el descenso durante varios años;
- en el **detrítico**, el hueco de P07 a principios de 2023 es un **periodo sin datos**, no un cambio real.

## 22. Hidrograma: precipitación y nivel

Para relacionar la lluvia con el nivel se usan dos paneles alineados en el tiempo: la precipitación arriba (barras) y el nivel abajo.

> 💡 A veces se dibujan ambas variables en el mismo panel con dos ejes y distintos (`twinx`). Es mejor evitarlo: con dos escalas, la posición relativa de las curvas depende de cómo se elijan los límites, y el lector puede ver relaciones que no existen. Dos paneles alineados comunican lo mismo sin ambigüedad.

In [ ]:
lluvia_e2 = precipitacion.loc[precipitacion["id_estacion"] == "E2"]

fig, (ax_p, ax_n) = plt.subplots(2, 1, figsize=(11, 6), sharex=True, gridspec_kw={"height_ratios": [1, 2]})

ax_p.bar(lluvia_e2["fecha"], lluvia_e2["precipitacion_mm"], width=20, color="#52514e")
ax_p.set_ylabel("Precipitación\n(mm/mes)")
ax_p.set_title("Estación E2 (Sierra) y pozo P05 (carbonatado)", loc="left")
ax_p.grid(alpha=0.3, axis="y")

ax_n.plot(p05["fecha"], p05["profundidad_nivel_m"], color=COLORES_ACUIFERO["Carbonatado"], linewidth=1.8)
ax_n.invert_yaxis()
ax_n.set_ylabel("Profundidad del nivel (m)")
ax_n.set_xlabel("Fecha")
ax_n.grid(alpha=0.3)

fig.tight_layout()
plt.show()

Fíjate en que el nivel **no** sube el mismo mes en que llueve: responde con cierto **retraso**, porque el agua necesita tiempo para infiltrarse y llegar al acuífero. En M8_8.27 mediremos ese retraso.

## 23. Diagrama de dispersión

Un diagrama de dispersión muestra la relación entre dos variables. Primero lo dibujamos con **todos** los datos:

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
for acuifero, grupo in datos.groupby("acuifero"):
    ax.scatter(grupo["profundidad_nivel_m"], grupo["conductividad_uScm"], s=12, alpha=0.7,
               color=COLORES_ACUIFERO[acuifero], label=acuifero)
ax.set_xlabel("Profundidad del nivel (m)")
ax.set_ylabel("Conductividad eléctrica (µS/cm)")
ax.legend(title="Acuífero")
ax.grid(alpha=0.3)
plt.show()

Un solo punto a 270 m **aplasta** todo el gráfico, y otro aparece con conductividad casi cero. Los gráficos también sirven para **encontrar errores**. Repetimos la figura sin los dos valores imposibles:

In [ ]:
es_plausible = (datos["profundidad_nivel_m"] < datos["profundidad_pozo_m"]) & (datos["conductividad_uScm"] > 50)

fig, ax = plt.subplots(figsize=(7, 5))
for acuifero, grupo in datos.loc[es_plausible].groupby("acuifero"):
    ax.scatter(grupo["profundidad_nivel_m"], grupo["conductividad_uScm"], s=12, alpha=0.7,
               color=COLORES_ACUIFERO[acuifero], label=acuifero)
ax.set_xlabel("Profundidad del nivel (m)")
ax.set_ylabel("Conductividad eléctrica (µS/cm)")
ax.legend(title="Acuífero")
ax.grid(alpha=0.3)
plt.show()

Ahora se distinguen los tres acuíferos y unos pocos puntos del aluvial con conductividad muy alta. ¿Son errores o un fenómeno real? Lo discutiremos en M8_8.27.

## 24. Histograma

Un histograma cuenta cuántos valores caen en cada intervalo (*bin*). Su forma depende del número de intervalos.

In [ ]:
profundidades_validas = datos.loc[es_plausible, "profundidad_nivel_m"].dropna()

fig, ejes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for ax, n_bins in zip(ejes, [8, 40]):
    ax.hist(profundidades_validas, bins=n_bins, color="#2a78d6", edgecolor="white")
    ax.set_title(f"{n_bins} intervalos")
    ax.set_xlabel("Profundidad del nivel (m)")
ejes[0].set_ylabel("Número de medidas")
fig.tight_layout()
plt.show()

Con 40 intervalos aparecen **tres grupos**, uno por acuífero. Con 8 intervalos, esa estructura casi desaparece. Un único histograma de todos los pozos mezcla sistemas distintos, y lo tendremos en cuenta en la estadística de M8_8.26.

## 25. Mapa de pozos

Las coordenadas UTM se pueden dibujar como un diagrama de dispersión. Con `set_aspect("equal")`, un metro en x mide lo mismo que un metro en y, como en un mapa. El color representa la cota piezométrica media, y la **barra de color** indica la escala.

In [ ]:
h_media = (datos.loc[es_plausible]
           .groupby("id_pozo")[["x_utm", "y_utm", "cota_piezometrica_m"]]
           .mean())

fig, ax = plt.subplots(figsize=(9, 5))
puntos = ax.scatter(h_media["x_utm"], h_media["y_utm"], c=h_media["cota_piezometrica_m"],
                    cmap="Blues", s=120, edgecolor="black", linewidth=0.5)
for id_pozo, fila in h_media.iterrows():
    ax.annotate(id_pozo, (fila["x_utm"], fila["y_utm"]), xytext=(6, 6), textcoords="offset points", fontsize=9)

fig.colorbar(puntos, ax=ax, label="Cota piezométrica media (m s.n.m.)")
ax.set_aspect("equal")
ax.set_xlabel("X UTM (m) · EPSG:25831")
ax.set_ylabel("Y UTM (m)")
ax.set_title("Cota piezométrica media 2021–2025")
ax.ticklabel_format(style="plain")
plt.show()

> 💡 **Conexión con otros bloques.** Con QGIS (M8_8.8) construisteis piezometrías a partir de puntos como estos. En M8_8.29 veréis cómo **interpolar** entre pozos.

## 26. Guardar una figura

`fig.savefig` guarda la figura. El nombre del archivo debe indicar su contenido. `dpi` controla la resolución y `bbox_inches="tight"` evita que se corten las etiquetas.

In [ ]:
ruta_figura = OUTPUT / "figures" / "mapa_cota_piezometrica_media.png"
fig.savefig(ruta_figura, dpi=200, bbox_inches="tight")
print("Guardada en", ruta_figura)

> ⚠️ **Decisiones que pueden inducir a error en una figura:**
> - recortar el eje y para exagerar una diferencia pequeña, o usar dos ejes y distintos;
> - unir con líneas puntos que no forman una secuencia;
> - omitir las unidades o el significado de los colores;
> - usar demasiados colores, o colores que no se distinguen (unas 1 de cada 12 personas tienen alguna alteración en la percepción del color);
> - pensar que un gráfico atractivo corrige unos datos incorrectos.
>
> Una buena figura responde a **una pregunta** y lleva título, ejes con unidades y leyenda.

## 27. Caso integrador · Ensayo de bombeo con el método de Cooper–Jacob

En el Módulo 1 estudiasteis la interpretación de ensayos de bombeo. Ahora vamos a usar lo que sabéis de NumPy, pandas y Matplotlib para hacer esa interpretación con código.

**Datos:** se bombea el pozo P02 a caudal constante $Q = 12$ L/s y se miden los descensos $s$ en un piezómetro situado a $r = 25$ m.

**Método de Cooper–Jacob.** Para tiempos suficientemente largos, la solución de Theis se aproxima a una recta en escala semilogarítmica:

$$
s = \frac{2.3\,Q}{4\pi T}\,\log_{10}\!\left(\frac{2.25\,T\,t}{r^2 S}\right)
$$

En un gráfico con $\log_{10} t$ en el eje x, los puntos forman una recta de pendiente $\Delta s$ (descenso por ciclo logarítmico). De la recta se obtienen los parámetros del acuífero:

$$
T = \frac{2.3\,Q}{4\pi\,\Delta s} \qquad\qquad S = \frac{2.25\,T\,t_0}{r^2}
$$

donde $t_0$ es el tiempo en el que la recta corta el eje $s = 0$.

La aproximación solo es válida si

$$
u = \frac{r^2 S}{4 T t} < 0.05
$$

por lo que los **primeros minutos** del ensayo no deben usarse en el ajuste.

**Unidades:** trabajaremos en metros y días, con $Q$ en m³/d, $t$ en d y $T$ en m²/d.

In [ ]:
ensayo = pd.read_csv(INPUT / "ensayo_bombeo_P02.csv")
ensayo["tiempo_d"] = ensayo["tiempo_min"] / 1440     # 1 día = 1440 minutos

Q_m3d = 12 * 86.4      # 12 L/s en m³/día
r_m = 25.0

fig, ax = plt.subplots(figsize=(8, 5))
ax.semilogx(ensayo["tiempo_min"], ensayo["descenso_m"], "o", color="#2a78d6", markersize=5)
ax.set_xlabel("Tiempo desde el inicio del bombeo (min, escala logarítmica)")
ax.set_ylabel("Descenso s (m)")
ax.set_title("Ensayo de bombeo en P02 · piezómetro a 25 m")
ax.grid(alpha=0.3, which="both")
plt.show()

A partir de unos 15 minutos, los puntos se alinean. Ajustamos una recta $s = a + b\,\log_{10} t$ **solo con esos puntos** usando `np.polyfit(x, y, 1)`, que devuelve la pendiente $b$ y la ordenada en el origen $a$.

> 📖 **Documentación:** https://numpy.org/doc/stable/reference/generated/numpy.polyfit.html. Fíjate en el **orden** en que devuelve los coeficientes.

In [ ]:
t_inicio_ajuste_min = 15
tramo = ensayo.loc[ensayo["tiempo_min"] >= t_inicio_ajuste_min]

pendiente, ordenada = np.polyfit(np.log10(tramo["tiempo_d"]), tramo["descenso_m"], 1)

delta_s = pendiente                                  # descenso por ciclo logarítmico (m)
T = 2.3 * Q_m3d / (4 * np.pi * delta_s)              # m²/d
t0_d = 10 ** (-ordenada / pendiente)                 # tiempo en que la recta corta s = 0
S = 2.25 * T * t0_d / r_m**2                         # adimensional

u_inicio = r_m**2 * S / (4 * T * t_inicio_ajuste_min / 1440)

print(f"Δs = {delta_s:.3f} m por ciclo logarítmico")
print(f"T  = {T:.0f} m²/d")
print(f"S  = {S:.1e}")
print(f"u al inicio del ajuste = {u_inicio:.3f}  → {'válido' if u_inicio < 0.05 else 'NO válido'} (u < 0.05)")

In [ ]:
t_recta = np.logspace(np.log10(t0_d), np.log10(ensayo["tiempo_d"].max()), 50)
s_recta = ordenada + pendiente * np.log10(t_recta)

fig, ax = plt.subplots(figsize=(8, 5))
ax.semilogx(ensayo["tiempo_min"], ensayo["descenso_m"], "o", color="#2a78d6", markersize=5, label="Medidas")
ax.semilogx(t_recta * 1440, s_recta, color="#eb6834", linewidth=2, label="Recta de Cooper–Jacob")
ax.axvline(t_inicio_ajuste_min, color="#52514e", linestyle="--", linewidth=1)
ax.text(t_inicio_ajuste_min * 1.1, 0.05, "inicio del ajuste", color="#52514e", fontsize=9)
ax.set_xlabel("Tiempo (min, escala logarítmica)")
ax.set_ylabel("Descenso s (m)")
ax.set_title(f"Cooper–Jacob: T ≈ {T:.0f} m²/d, S ≈ {S:.1e}")
ax.legend()
ax.grid(alpha=0.3, which="both")
fig.savefig(OUTPUT / "figures" / "ensayo_bombeo_P02_cooper_jacob.png", dpi=200, bbox_inches="tight")
plt.show()

**Interpretación.** Una transmisividad de unos 300 m²/d y un coeficiente de almacenamiento del orden de 10⁻³ son valores coherentes con un acuífero aluvial de gravas y arenas en condiciones de semiconfinamiento. El resultado depende del tramo elegido para el ajuste, de que el caudal haya sido realmente constante y de las hipótesis del método (acuífero homogéneo, isótropo y de extensión infinita).

### ✏️ Ejercicio 4 · ¿Importa el tramo de ajuste?

Repite el cálculo usando **todos** los puntos (`t_inicio = 1` minuto) y guarda los resultados en `T_todos` y `S_todos`.

1. ¿Cuánto cambian `T` y `S` respecto al ajuste desde 15 minutos?
2. Calcula `u` al primer minuto (`u_1min`). ¿Se cumple la condición de validez?
3. ¿Qué parámetro es más sensible a incluir los primeros puntos?

In [ ]:
t_inicio = 1
tramo_todos = ensayo.loc[ensayo["tiempo_min"] >= t_inicio]

pendiente_t, ordenada_t = np.polyfit(np.log10(tramo_todos["tiempo_d"]), tramo_todos["descenso_m"], 1)
T_todos = 2.3 * Q_m3d / (4 * np.pi * pendiente_t)
S_todos = 2.25 * T_todos * 10 ** (-ordenada_t / pendiente_t) / r_m**2
u_1min = r_m**2 * S_todos / (4 * T_todos * t_inicio / 1440)

print(f"T: {T:.0f} → {T_todos:.0f} m²/d | S: {S:.2e} → {S_todos:.2e} | u(1 min) = {u_1min:.2f}")

In [ ]:
# ✅ Comprobación
assert abs(T_todos - T) / T < 0.10, "T debería cambiar poco (menos de un 10 %)"
assert abs(S_todos - S) / S > 0.10, "S debería cambiar bastante más que T"
assert u_1min > 0.05
print("✔ Correcto. Incluir los primeros minutos, donde u > 0.05, altera sobre todo S,")
print("  porque S depende de dónde corta la recta el eje (t0), que es muy sensible a la pendiente.")

### Para ir más allá (opcional) · Comparar con la solución de Theis

La solución completa de Theis es $s = \frac{Q}{4\pi T}\,W(u)$, donde $W(u)$ es la **función de pozo**. En SciPy está disponible como `scipy.special.exp1` (📖 https://docs.scipy.org/doc/scipy/reference/generated/scipy.special.exp1.html). Ejecuta la celda para comparar la curva de Theis, calculada con los parámetros obtenidos, con las medidas.

In [ ]:
from scipy.special import exp1

t_curva = np.logspace(np.log10(ensayo["tiempo_d"].min()), np.log10(ensayo["tiempo_d"].max()), 100)
u_curva = r_m**2 * S / (4 * T * t_curva)
s_theis = Q_m3d / (4 * np.pi * T) * exp1(u_curva)

fig, ax = plt.subplots(figsize=(8, 5))
ax.semilogx(ensayo["tiempo_min"], ensayo["descenso_m"], "o", color="#2a78d6", markersize=5, label="Medidas")
ax.semilogx(t_curva * 1440, s_theis, color="#1baf7a", linewidth=2, label="Theis (T y S de Cooper–Jacob)")
ax.semilogx(t_recta * 1440, s_recta, color="#eb6834", linewidth=1.5, linestyle="--", label="Recta de Cooper–Jacob")
ax.set_xlabel("Tiempo (min, escala logarítmica)")
ax.set_ylabel("Descenso s (m)")
ax.legend()
ax.grid(alpha=0.3, which="both")
plt.show()

## Síntesis

- **NumPy:** los arrays permiten calcular sin bucles; las máscaras seleccionan y cuentan; `nan…` ignora ausentes; `axis` indica en qué dirección se resume una matriz.
- **pandas:** siempre *leer → inspeccionar (`info`, `describe`) → seleccionar/filtrar → unir → agrupar → guardar*. Después de unir tablas, comprueba que no se ha perdido ni quedado huérfana ninguna fila.
- **Matplotlib:** `fig, ax = plt.subplots()`, ejes con unidades, un color fijo por entidad, paneles alineados en lugar de dos ejes y distintos, y la galería como fuente de ejemplos.
- Los gráficos y `describe()` son también herramientas de **control de calidad**: hoy hemos encontrado un valor imposible (270 m), un error de unidades (0.97 µS/cm), un identificador mal escrito (`P1`), duplicados y un hueco de datos.

### Lo que viene en M8_8.25–27

Organizaremos estos datos en una **base de datos relacional**, describiremos su distribución con **estadística** y trataremos de forma sistemática todos los problemas de **calidad** que hemos anotado.